In [1]:
import os
os.chdir('G:\AKT1\Docking')

In [2]:
os.getcwd()

'G:\\AKT1\\Docking'

In [ ]:

import os
import csv
import subprocess

# ─────────────────────────────────────────────────────────────
# Settings
# ─────────────────────────────────────────────────────────────
VINA_EXE       = "vina"
CONFIG_FILE    = "config.txt"    # must set receptor + center_x/y/z + size_x/y/z
INPUT_DIR      = "Processed_Ligand"
OUTPUT_DIR     = "Docked_Out"
TOP_N          = 10
EXHAUSTIVENESS = 8
SEED           = 42              # fixed seed -> reproducible poses; report this value in methods

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(CONFIG_FILE):
    raise FileNotFoundError(
        f"{CONFIG_FILE} not found. Before running, confirm center_x/y/z in "
        "it points at the grid center you validated during redocking, not "
        "just whichever pocket center fpocket reported."
    )


# ─────────────────────────────────────────────────────────────
# Vina execution
# ─────────────────────────────────────────────────────────────
def run_vina(ligand_path, output_file, log_file):
    """Run one Vina docking job. Returns True on success, False on failure."""
    cmd = [
        VINA_EXE, "--config", CONFIG_FILE,
        "--ligand", ligand_path,
        "--out", output_file,
        "--log", log_file,
        "--exhaustiveness", str(EXHAUSTIVENESS),
        "--seed", str(SEED),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"    Vina failed (exit {result.returncode}): "
              f"{result.stderr.strip()[:200]}")
        return False
    if not os.path.exists(log_file):
        print("    No log file produced.")
        return False
    return True


def best_affinity(log_file):
    """
    Parse the best (mode 1) binding affinity from a Vina log file.
    Mode 1's row comes immediately after the dashed separator line,
    with no blank row in between, so the very next line is the value
    we want. No extra next(f) here.
    """
    with open(log_file, errors="ignore") as f:
        for line in f:
            if line.startswith("-----+------------+----------+----------"):
                affinity_line = next(f, None)
                if affinity_line is None:
                    return None
                try:
                    return float(affinity_line.split()[1])
                except (IndexError, ValueError):
                    return None
    return None


# ─────────────────────────────────────────────────────────────
# Batch docking
# ─────────────────────────────────────────────────────────────
ligands = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith(".pdbqt"))
print(f"Docking {len(ligands)} ligands with Vina "
      f"(exhaustiveness={EXHAUSTIVENESS}, seed={SEED})...\n")

results = []   # (ligand_name, affinity)
failed  = []   # ligand names that failed or couldn't be parsed

for i, ligand in enumerate(ligands, 1):
    ligand_name = ligand.rsplit(".", 1)[0]
    ligand_path = os.path.join(INPUT_DIR, ligand)
    output_file = os.path.join(OUTPUT_DIR, f"{ligand_name}_out.pdbqt")
    log_file    = os.path.join(OUTPUT_DIR, f"{ligand_name}_log.txt")

    # Resume support: reuse a completed docking from a previous run
    if os.path.exists(output_file) and os.path.exists(log_file):
        affinity = best_affinity(log_file)
        if affinity is not None:
            results.append((ligand_name, affinity))
            print(f"  [{i}/{len(ligands)}] {ligand_name}: already done, "
                  f"affinity {affinity:.2f}")
            continue

    print(f"  [{i}/{len(ligands)}] Docking {ligand_name} ...")
    if not run_vina(ligand_path, output_file, log_file):
        failed.append(ligand_name)
        continue

    affinity = best_affinity(log_file)
    if affinity is None:
        print(f"    Could not parse affinity from {log_file}")
        failed.append(ligand_name)
        continue

    results.append((ligand_name, affinity))
    print(f"    {ligand_name}: {affinity:.2f} kcal/mol")

print(f"\nDocked {len(results)} / {len(ligands)}  "
      f"({len(failed)} failed" + (f": {failed}" if failed else "") + ")")

# ─────────────────────────────────────────────────────────────
# Save full ranked results + top N
# ─────────────────────────────────────────────────────────────
results.sort(key=lambda x: x[1])   # most negative affinity = best, first

full_csv = os.path.join(OUTPUT_DIR, "all_docking_results.csv")
with open(full_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Ligand", "Affinity_kcal_mol", "Rank",
                      "Exhaustiveness", "Seed"])
    for rank, (name, aff) in enumerate(results, 1):
        writer.writerow([name, f"{aff:.2f}", rank, EXHAUSTIVENESS, SEED])
print(f"\nFull ranked results: {full_csv}")

top_molecules = results[:TOP_N]
print(f"\nTop {min(TOP_N, len(results))} molecules:")
for i, (name, aff) in enumerate(top_molecules, 1):
    print(f"  {i}. {name}: {aff:.2f}")

top_file = os.path.join(OUTPUT_DIR, "top_molecules.txt")
with open(top_file, "w") as f:
    f.write("\n".join(f"{name}: {aff:.2f}" for name, aff in top_molecules))
print(f"\nTop {TOP_N} list saved: {top_file}")

if failed:
    failed_file = os.path.join(OUTPUT_DIR, "failed_ligands.txt")
    with open(failed_file, "w") as f:
        f.write("\n".join(failed))
    print(f"Failed ligands logged: {failed_file}")